<div style="background-color: #f0f0f0; color:rgb(56, 56, 56); padding: 8px; display: flex; align-items: center; gap: 100px;">
  <img src="../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI<br/>
    3.Lab.2 Advanced RNN Variants for Audio Processing<br/>
  </strong>
  <emph>
    Student Name: [Complete with your name]<br/>
    Date: [Fill up with the submission date]<br/>
  </emph>
</div>

## Background

This lab focuses on implementing and comparing advanced RNN architectures (LSTM and GRU) for audio processing tasks. You'll gain insight into how these variants address the limitations of vanilla RNNs through their sophisticated gating mechanisms. By exploring bidirectional processing, attention mechanisms, and gate dynamics, you'll understand the architectural innovations that make modern sequence models powerful.

---

## Description

During this lab session, you will:

- Implement LSTM and GRU models using PyTorch
- Compare their performance with vanilla RNNs on audio classification
- Experiment with bidirectional variants of each architecture
- Analyze how each model handles long audio sequences
- Implement attention mechanisms to improve performance
- Visualize gate activations in LSTM/GRU cells during inference
- Explore impact of hyperparameters on different architectures
- Assess computational efficiency vs. performance trade-offs

**Prerequisites:** You should have completed Lab 1 (RNN Implementation for Audio Classification) before starting this lab.

**Key Deliverables:**
- Complete this notebook with all code cells executed
- Working implementations of LSTM and GRU models
- Comparative analysis of different architectures
- Visualizations of gate dynamics and attention weights
- Performance metrics across different sequence lengths
- Submit the notebook with all outputs visible

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torchaudio
import torchaudio.transforms as T
from torchaudio.datasets import SPEECHCOMMANDS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tqdm import tqdm
import time
import os
import warnings

from rich.console import Console
from rich.table import Table

console = Console()

warnings.filterwarnings("ignore")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
console.print(f"Using device: {device}")
if device.type == "cuda":
    console.print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Setup and Data Loading

Reuse the dataset code from Lab 1.

In [ ]:
# Reuse the dataset code from Lab 1
COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]


class SubsetSC(SPEECHCOMMANDS):
    """Filters the Speech Commands dataset to only include specified labels."""

    def __init__(self, subset: str = None, commands: list = None):
        super().__init__(
            "/content/drive/MyDrive/Laboratories/data/", download=True, subset=subset
        )

        if commands:

            def load_list(filename):
                filepath = os.path.join(self._path, filename)
                with open(filepath) as fileobj:
                    return [
                        os.path.normpath(os.path.join(self._path, line.strip()))
                        for line in fileobj
                    ]

            if subset == "validation":
                self._walker = load_list("validation_list.txt")
            elif subset == "testing":
                self._walker = load_list("testing_list.txt")
            elif subset == "training":
                excludes = load_list("validation_list.txt") + load_list(
                    "testing_list.txt"
                )
                excludes = set(excludes)
                self._walker = [w for w in self._walker if w not in excludes]

            self._walker = [w for w in self._walker if w.split(os.sep)[-2] in commands]


# Load datasets
train_set = SubsetSC(subset="training", commands=COMMANDS)
val_set = SubsetSC(subset="validation", commands=COMMANDS)
test_set = SubsetSC(subset="testing", commands=COMMANDS)

console.print(
    f"Training: {len(train_set)}, Validation: {len(val_set)}, Test: {len(test_set)}"
)

In [ ]:
# Feature extraction (MFCC)
mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40,
    melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 64},
)


# Dataset class from Lab 1
class AudioDataset(Dataset):
    def __init__(self, subset_dataset, transform, label_to_idx, max_length=101):
        self.dataset = subset_dataset
        self.transform = transform
        self.label_to_idx = label_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        waveform, sample_rate, label, *_ = self.dataset[idx]
        features = self.transform(waveform).squeeze(0).T  # (time, features)

        # Pad or truncate
        if features.shape[0] < self.max_length:
            pad = torch.zeros(self.max_length - features.shape[0], features.shape[1])
            features = torch.cat([features, pad], dim=0)
        else:
            features = features[: self.max_length]

        label_idx = self.label_to_idx[label]
        return features, label_idx


# Create datasets
label_to_idx = {label: idx for idx, label in enumerate(COMMANDS)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

train_dataset = AudioDataset(train_set, mfcc_transform, label_to_idx)
val_dataset = AudioDataset(val_set, mfcc_transform, label_to_idx)
test_dataset = AudioDataset(test_set, mfcc_transform, label_to_idx)

# Data loaders
batch_size = 32
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

## Part 2: LSTM Implementation

Let's implement an LSTM model. LSTMs address the vanishing gradient problem through their gating mechanisms.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout=0.3,
        bidirectional=False,
    ):
        """
        Args:
            input_size: Number of input features
            hidden_size: Number of hidden units
            num_layers: Number of LSTM layers
            num_classes: Number of output classes
            dropout: Dropout probability
            bidirectional: Whether to use bidirectional LSTM
        """
        super(LSTMClassifier, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
        )

        self.dropout = nn.Dropout(dropout)

        fc_input_size = hidden_size * 2 if bidirectional else hidden_size
        self.fc = nn.Linear(fc_input_size, num_classes)

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

    def forward(self, x):
        """
        Args:
            x: Input tensor (batch_size, seq_length, input_size)
        Returns:
            out: Output logits (batch_size, num_classes)
            (h_n, c_n): Final hidden and cell states
        """

        lstm_out, (h_n, c_n) = self.lstm(x)

        if self.bidirectional:
            out = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            out = h_n[-1]

        out = self.dropout(out)

        out = self.fc(out)

        return out, (h_n, c_n)


# Instantiate LSTM model
input_size = 40
hidden_size = 128
num_layers = 2
num_classes = len(COMMANDS)

lstm_model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes).to(device)

console.print(lstm_model)
console.print(
    f"\nTotal parameters: {sum(p.numel() for p in lstm_model.parameters()):,}"
)

#### REFLECTION QUESTION 1:

Explain the three gates in an LSTM cell (input gate, forget gate, output gate). How do they help address the vanishing gradient problem?

**Your Answer:**

The input gate controls what part of the new information enters the memory. The forget gate decides what to forget, so as not to accumulate things that are no longer useful. And the output gate chooses what part of that memory is used in the final output.

Thanks to this mechanism, the model does not get stuck with only the last thing it heard, but can retain important information for longer. This helps avoid the typical problem with RNNs where gradients are lost and the model forgets things quickly.

## Part 3: GRU Implementation

Now let's implement a GRU model. GRUs are a simplified variant of LSTMs with fewer parameters.

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout=0.3,
        bidirectional=False,
    ):
        """
        Args:
            input_size: Number of input features
            hidden_size: Number of hidden units
            num_layers: Number of GRU layers
            num_classes: Number of output classes
            dropout: Dropout probability
            bidirectional: Whether to use bidirectional GRU
        """
        super(GRUClassifier, self).__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
        )

        self.dropout = nn.Dropout(dropout)

        fc_input_size = hidden_size * 2 if bidirectional else hidden_size
        self.fc = nn.Linear(fc_input_size, num_classes)

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

    def forward(self, x):
        """
        Args:
            x: Input tensor (batch_size, seq_length, input_size)
        Returns:
            out: Output logits (batch_size, num_classes)
            h_n: Final hidden state
        """

        gru_out, h_n = self.gru(x)

        if self.bidirectional:
            out = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            out = h_n[-1]

        out = self.dropout(out)
        out = self.fc(out)

        return out, h_n


# Instantiate GRU model
gru_model = GRUClassifier(input_size, hidden_size, num_layers, num_classes).to(device)

console.print(gru_model)
console.print(f"\nTotal parameters: {sum(p.numel() for p in gru_model.parameters()):,}")

#### REFLECTION QUESTION 2:

Compare the architecture of GRU with LSTM. What are the main differences? What are the trade-offs between using GRU vs LSTM?

**Your Answer:**

An LSTM and a GRU seek to solve the same problem: learning patterns in sequential data without forgetting important information. The difference is that the LSTM has three gates (input, forget, and output) and handles two internal states, while the GRU is simpler, combining those gates into just two and working with a single state. In practice, LSTM can capture longer relationships, but it is heavier and takes longer to train. GRU is lighter, trains faster, and sometimes performs equally well or even better depending on the dataset.

## Part 4: Training Framework

Let's create a flexible training framework to train and compare different models.

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    num_epochs,
    device,
    model_name="model",
):
    """
    Train a model and track performance metrics.

    Returns:
        history: Dictionary containing training history
    """
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "train_time": [],
    }

    best_val_acc = 0.0

    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        start_time = time.time()

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        epoch_time = time.time() - start_time
        train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs, _ = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["train_time"].append(epoch_time)

        console.print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        console.print(
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Time: {epoch_time:.2f}s"
        )

    console.print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    return history

## Part 5: Model Training and Comparison

Now let's train all models and compare their performance.

In [ ]:
# Training configuration
num_epochs = 20
learning_rate = 0.001

# Store all model histories for comparison
all_histories = {}

In [ ]:
console.print("=" * 60)
console.print("Training LSTM Model")
console.print("=" * 60)

lstm_model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes).to(device)
lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=learning_rate)
lstm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    lstm_optimizer, mode="min", factor=0.5, patience=3
)

lstm_history = train_model(
    lstm_model,
    train_loader,
    val_loader,
    lstm_criterion,
    lstm_optimizer,
    lstm_scheduler,
    num_epochs,
    device,
    model_name="lstm",
)

all_histories["LSTM"] = lstm_history

In [ ]:
console.print("\n" + "=" * 60)
console.print("Training GRU Model")
console.print("=" * 60)

gru_model = GRUClassifier(input_size, hidden_size, num_layers, num_classes).to(device)

gru_criterion = nn.CrossEntropyLoss()
gru_optimizer = optim.Adam(gru_model.parameters(), lr=learning_rate)
gru_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    gru_optimizer, mode="min", factor=0.5, patience=3
)

gru_history = train_model(
    gru_model,
    train_loader,
    val_loader,
    gru_criterion,
    gru_optimizer,
    gru_scheduler,
    num_epochs,
    device,
    model_name="gru",
)

all_histories["GRU"] = gru_history

### 5.1 Performance Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for model_name, history in all_histories.items():
    axes[0, 0].plot(history["train_loss"], label=model_name, marker="o", markersize=4)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

for model_name, history in all_histories.items():
    axes[0, 1].plot(history["val_loss"], label=model_name, marker="s", markersize=4)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].set_title("Validation Loss")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

for model_name, history in all_histories.items():
    axes[1, 0].plot(history["train_acc"], label=model_name, marker="o", markersize=4)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].set_title("Training Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

for model_name, history in all_histories.items():
    axes[1, 1].plot(history["val_acc"], label=model_name, marker="s", markersize=4)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].set_title("Validation Accuracy")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

In [ ]:
comparison_data = []

for model_name, history in all_histories.items():
    comparison_data.append(
        {
            "Model": model_name,
            "Best Val Acc": f"{max(history['val_acc']):.4f}",
            "Final Train Acc": f"{history['train_acc'][-1]:.4f}",
            "Final Val Acc": f"{history['val_acc'][-1]:.4f}",
            "Avg Epoch Time (s)": f"{np.mean(history['train_time']):.2f}",
        }
    )

comparison_df = pd.DataFrame(comparison_data)
console.print("\nModel Performance Comparison:")
console.print(comparison_df.to_string(index=False))

#### REFLECTION QUESTION 3:

Based on the training curves and comparison table, which architecture performed best? Why do you think this is the case?

**Your Answer:**

The GRU performed best overall because it learned faster in curves, remained stable, and ended up with higher validation accuracy than the LSTM. Being a simpler architecture, it does not need as much complexity to capture audio patterns, so it avoids overfitting and responds better in validation. In short, the GRU found a good balance between performance and stability, which is why it ended up delivering the best results.

## Part 6: Bidirectional RNNs

Let's explore bidirectional variants, which process sequences in both forward and backward directions.

In [ ]:
console.print("=" * 60)
console.print("Training Bidirectional LSTM Model")
console.print("=" * 60)

bilstm_model = LSTMClassifier(
    input_size, hidden_size, num_layers, num_classes, bidirectional=True
).to(device)

bilstm_criterion = nn.CrossEntropyLoss()
bilstm_optimizer = optim.Adam(bilstm_model.parameters(), lr=learning_rate)
bilstm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    bilstm_optimizer, mode="min", factor=0.5, patience=3
)

bilstm_history = train_model(
    bilstm_model,
    train_loader,
    val_loader,
    bilstm_criterion,
    bilstm_optimizer,
    bilstm_scheduler,
    num_epochs,
    device,
    model_name="bilstm",
)

all_histories["BiLSTM"] = bilstm_history

In [ ]:
console.print("\n" + "=" * 60)
console.print("Training Bidirectional GRU Model")
console.print("=" * 60)

bigru_model = GRUClassifier(
    input_size, hidden_size, num_layers, num_classes, bidirectional=True
).to(device)

bigru_criterion = nn.CrossEntropyLoss()
bigru_optimizer = optim.Adam(bigru_model.parameters(), lr=learning_rate)
bigru_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    bigru_optimizer, mode="min", factor=0.5, patience=3
)

bigru_history = train_model(
    bigru_model,
    train_loader,
    val_loader,
    bigru_criterion,
    bigru_optimizer,
    bigru_scheduler,
    num_epochs,
    device,
    model_name="bigru",
)

all_histories["BiGRU"] = bigru_history

#### REFLECTION QUESTION 4:

When is using bidirectional RNNs beneficial? Are there any scenarios where bidirectional RNNs would NOT be appropriate?

**Your Answer:**

Using bidirectional RNNs is useful when we need to understand sequential data from both the past and the future, for example in tasks where having the full context completes the meaning, analysis of pre-recorded audio, transcription, or any signal where looking "ahead" helps to better interpret what happened before. However, they are not always appropriate, if the system works in real time and we must process information as it arrives (such as live voice recognition or event detection as they occur), we cannot use a bidirectional model because it needs to see the entire sequence before producing an output. In such cases, we must use unidirectional models that work without waiting for the future.

## Part 7: Final Evaluation on Test Set

Let's evaluate our best models on the held-out test set.

In [ ]:
def evaluate_model(model, test_loader, device, model_name):
    model.eval()
    all_preds = []
    all_labels = []
    inference_times = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=f"Evaluating {model_name}"):
            inputs = inputs.to(device)

            start_time = time.time()
            outputs, _ = model(inputs)
            inference_times.append(time.time() - start_time)

            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    avg_inference_time = np.mean(inference_times) * 1000

    console.print(f"\n{model_name} Test Results:")
    console.print(f"Accuracy: {accuracy:.4f}")
    console.print(f"Avg Inference Time: {avg_inference_time:.2f}ms")
    console.print("\nClassification Report:")
    console.print(classification_report(all_labels, all_preds, target_names=COMMANDS))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=COMMANDS,
        yticklabels=COMMANDS,
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

    return accuracy, avg_inference_time

In [ ]:
results = {}

model_dict = {
    "LSTM": lstm_model,
    "GRU": gru_model,
    "BiLSTM": bilstm_model,
    "BiGRU": bigru_model,
}

for model_name, model in model_dict.items():
    model.load_state_dict(torch.load(f"best_{model_name.lower()}.pth"))
    accuracy, inf_time = evaluate_model(model, test_loader, device, model_name)

    num_params = sum(p.numel() for p in model.parameters())
    results[model_name] = {
        "Test Accuracy": f"{accuracy:.4f}",
        "Inference Time (ms)": f"{inf_time:.2f}",
        "Parameters": f"{num_params:,}",
    }

In [ ]:
table = Table(title="Final Model Comparison", show_lines=True)

table.add_column("Model", style="bold cyan")
table.add_column("Test Accuracy", justify="center")
table.add_column("Inference Time (ms)", justify="center")
table.add_column("Parameters", justify="right")

for model_name, metrics in results.items():
    table.add_row(
        model_name,
        metrics["Test Accuracy"],
        metrics["Inference Time (ms)"],
        metrics["Parameters"],
    )

console.print(table)

## Part 8: Analysis and Insights

#### REFLECTION QUESTION 6:

Summarize the key differences in performance between vanilla RNN, LSTM, and GRU. Which would you choose for a production audio classification system and why?

**Your Answer:**

In general, basic RNNs tend to fall short because they do not handle long sequences well and their performance is more unstable. LSTM improves significantly because it remembers information better, but it is usually heavier and slower to train. GRU, on the other hand, strikes a good balance: it learns patterns in audio well, trains faster, and uses fewer resources. Therefore, for a production system, I would choose GRU because it offers solid performance without overcomplicating the model or increasing the computational cost too much.

#### REFLECTION QUESTION 7:

Discuss the trade-offs between model complexity, training time, inference speed, and accuracy. How would these trade-offs influence your architecture choice for:
- A real-time voice assistant on a mobile device
- An offline batch processing system for audio analysis
- A cloud-based API for audio classification

**Your Answer:**

When building a model, you always have to balance how complex it is, how long it takes to train, how fast it responds, and how accurate it is. That balance changes depending on the scenario.
For a real-time voice assistant on a cell phone, the ideal is something lightweight and fast, even if it's not the most accurate model in the world; a small GRU works well there.
In an offline system that processes lots of audio in batches, you can use a larger, more accurate model because response time is not as important.
And in a cloud API, you look for a middle ground: good accuracy, but with reasonable speed so as not to keep the user waiting and without driving up computing costs.

#### REFLECTION QUESTION 8:

What are some potential improvements or extensions you could make to these models? Consider:
- Different audio features (raw waveforms, different spectral representations)
- Hybrid architectures (CNNs + RNNs)
- Data augmentation techniques
- Advanced attention mechanisms

**Your Answer:**

The model can always be improved without making it too complicated. For example, other ways of representing audio can be tried, not just MFCC, to see if the model captures certain details better. Architectures can also be mixed, such as using a CNN at the beginning to extract clearer patterns and then an RNN that understands the temporal part. Another option is to apply data augmentation, basically "varying" the audio (noise, speed changes, etc.) so that the model learns to be more robust. And if you want to go a step further, you could use attention mechanisms so that the model learns to focus on the most important parts of the sound instead of treating everything equally.

---

## Conclusion

Congratulations! You've successfully:
- Implemented LSTM and GRU models for audio classification
- Compared vanilla RNN, LSTM, and GRU performance
- Explored bidirectional variants
- Implemented attention mechanisms
- Analyzed computational efficiency vs. performance trade-offs
- Evaluated models on held-out test data

### Key Takeaways

1. **LSTM and GRU** effectively address the vanishing gradient problem in vanilla RNNs
2. **Bidirectional RNNs** can improve performance when full sequence context is available
3. **Attention mechanisms** help models focus on relevant parts of sequences
4. **Architecture choice** involves trade-offs between accuracy, complexity, and efficiency
5. **Audio classification** benefits from sequential modeling of temporal patterns

These skills will be valuable as you move on to more advanced sequence modeling architectures like Transformers!